<cell_type>markdown</cell_type># TensorRT 推理教程 (TensorRT Inference Tutorial)

> **前置知识**: PyTorch 基础、ONNX 模型格式、NVIDIA GPU 基础
>
> **学习目标**: 掌握 TensorRT 引擎构建和高性能 GPU 推理

---

## 什么是 TensorRT？

```
┌─────────────────────────────────────────────────────────────┐
│                    TensorRT 优化流程                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ONNX/PyTorch Model                                         │
│         │                                                   │
│         ▼                                                   │
│  ┌─────────────────────────────────────────────────────┐   │
│  │              TensorRT Builder                        │   │
│  │  ┌─────────────────────────────────────────────┐    │   │
│  │  │  1. 层融合 (Layer Fusion)                   │    │   │
│  │  │     Conv + BN + ReLU → ConvBNReLU          │    │   │
│  │  │                                             │    │   │
│  │  │  2. 精度校准 (Calibration)                  │    │   │
│  │  │     FP32 → FP16/INT8                       │    │   │
│  │  │                                             │    │   │
│  │  │  3. 内核自动调优 (Auto-Tuning)              │    │   │
│  │  │     选择最优 CUDA 内核实现                  │    │   │
│  │  │                                             │    │   │
│  │  │  4. 内存优化                                │    │   │
│  │  │     张量复用、内存池                        │    │   │
│  │  └─────────────────────────────────────────────┘    │   │
│  └─────────────────────────────────────────────────────┘   │
│         │                                                   │
│         ▼                                                   │
│  TensorRT Engine (.engine/.plan)                           │
│  (序列化的优化模型，可直接加载推理)                         │
│                                                             │
│  核心优势:                                                  │
│  - NVIDIA GPU 极致性能优化                                 │
│  - 支持 FP32/FP16/INT8 多种精度                            │
│  - 动态形状支持                                            │
│  - 比 PyTorch 快 2-10x                                     │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

**注意**: TensorRT 仅支持 NVIDIA GPU

## 本教程内容

1. **TensorRT 核心概念** - 优化技术和精度模式
2. **引擎构建** - 从 ONNX 构建 TensorRT 引擎
3. **动态形状** - 支持可变输入大小
4. **推理执行** - 使用引擎进行推理
5. **性能对比** - PyTorch vs TensorRT

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import numpy as np
import os
import tempfile
import time

# 设置随机种子
np.random.seed(42)

print("=" * 60)
print("环境检查")
print("=" * 60)

# 检查 TensorRT
try:
    import tensorrt as trt
    TENSORRT_AVAILABLE = True
    print(f"\n✓ TensorRT 版本: {trt.__version__}")
except ImportError:
    TENSORRT_AVAILABLE = False
    print("\n✗ TensorRT 未安装")
    print("  安装命令: pip install tensorrt")

# 检查 PyCUDA
try:
    import pycuda.driver as cuda
    import pycuda.autoinit
    PYCUDA_AVAILABLE = True
    print(f"✓ PyCUDA 可用")
except ImportError:
    PYCUDA_AVAILABLE = False
    print("✗ PyCUDA 未安装")
    print("  安装命令: pip install pycuda")

# 检查 PyTorch
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    PYTORCH_AVAILABLE = True
    print(f"✓ PyTorch 版本: {torch.__version__}")
    print(f"  CUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    PYTORCH_AVAILABLE = False
    print("✗ PyTorch 未安装")

# 总体状态
print("\n" + "=" * 60)
if TENSORRT_AVAILABLE and PYCUDA_AVAILABLE:
    print("✓ TensorRT 环境就绪，可以运行完整示例")
else:
    print("! TensorRT 环境不完整，部分示例将跳过")
    print("  本教程仍可学习 TensorRT 概念和 API")

<cell_type>markdown</cell_type>## 1. TensorRT 核心优化技术

**核心概念**: TensorRT 通过多种优化技术实现极致 GPU 性能

```
┌─────────────────────────────────────────────────────────────┐
│                    TensorRT 优化技术                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 层融合 (Layer Fusion)                                   │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  优化前:              优化后:                       │   │
│  │  ┌─────┐             ┌─────────────┐               │   │
│  │  │Conv │             │             │               │   │
│  │  └──┬──┘             │ Conv+BN+ReLU│               │   │
│  │     │                │  (融合内核)  │               │   │
│  │  ┌──▼──┐             │             │               │   │
│  │  │ BN  │     →       └─────────────┘               │   │
│  │  └──┬──┘                                           │   │
│  │     │                内存访问: 3次 → 1次           │   │
│  │  ┌──▼──┐             内核启动: 3次 → 1次           │   │
│  │  │ReLU │                                           │   │
│  │  └─────┘                                           │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  2. 内核自动调优 (Auto-Tuning)                              │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  针对特定 GPU 架构选择最优 CUDA 内核实现            │   │
│  │  - 不同卷积算法 (Winograd, FFT, Direct)            │   │
│  │  - 不同 tile 大小                                  │   │
│  │  - 不同内存访问模式                                │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  3. 精度模式                                                │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  精度    位数    速度      精度损失    适用场景     │   │
│  │  ─────   ────    ────      ────────    ────────     │   │
│  │  FP32    32      1x        无          训练/调试    │   │
│  │  FP16    16      2x        极小        生产推理     │   │
│  │  INT8    8       4x        需要校准    高吞吐场景   │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

<cell_type>markdown</cell_type>## 2. 创建测试模型并导出 ONNX

**核心概念**: TensorRT 通常从 ONNX 模型构建引擎

```
┌─────────────────────────────────────────────────────────────┐
│                    TensorRT 构建流程                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  PyTorch Model                                              │
│       │                                                     │
│       ▼ torch.onnx.export()                                │
│  ONNX Model (.onnx)                                         │
│       │                                                     │
│       ▼ TensorRT Builder                                   │
│  TensorRT Engine (.engine)                                  │
│       │                                                     │
│       ▼ TensorRT Runtime                                   │
│  推理执行                                                   │
│                                                             │
│  关键步骤:                                                  │
│  1. 创建 Logger (日志记录)                                 │
│  2. 创建 Builder (引擎构建器)                              │
│  3. 创建 Network (网络定义)                                │
│  4. 解析 ONNX (OnnxParser)                                 │
│  5. 配置优化选项 (BuilderConfig)                           │
│  6. 构建序列化引擎                                         │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 定义测试模型
# ============================================================

if PYTORCH_AVAILABLE:
    class SimpleConvNet(nn.Module):
        """
        简单卷积网络
        
        用于演示 TensorRT 引擎构建和推理
        
        结构:
        - 2 个卷积层 (带 BatchNorm)
        - 2 个全连接层
        """
        def __init__(self, num_classes=10):
            super().__init__()
            # 卷积层 1: 3 通道 → 32 通道
            self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
            self.bn1 = nn.BatchNorm2d(32)
            
            # 卷积层 2: 32 通道 → 64 通道
            self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
            self.bn2 = nn.BatchNorm2d(64)
            
            # 池化层
            self.pool = nn.MaxPool2d(2)
            
            # 全连接层
            self.fc1 = nn.Linear(64 * 8 * 8, 256)
            self.fc2 = nn.Linear(256, num_classes)
        
        def forward(self, x):
            # 卷积块 1: Conv → BN → ReLU → Pool
            x = self.pool(torch.relu(self.bn1(self.conv1(x))))  # [B,3,32,32] → [B,32,16,16]
            
            # 卷积块 2: Conv → BN → ReLU → Pool
            x = self.pool(torch.relu(self.bn2(self.conv2(x))))  # [B,32,16,16] → [B,64,8,8]
            
            # 展平
            x = x.view(x.size(0), -1)  # [B,64,8,8] → [B,4096]
            
            # 全连接层
            x = torch.relu(self.fc1(x))  # [B,4096] → [B,256]
            return self.fc2(x)  # [B,256] → [B,10]
    
    # 创建模型并设置为评估模式
    model = SimpleConvNet()
    model.eval()
    
    # 统计参数
    total_params = sum(p.numel() for p in model.parameters())
    
    print("=" * 60)
    print("测试模型信息")
    print("=" * 60)
    print(f"\n模型结构:")
    print(f"  输入: [B, 3, 32, 32]")
    print(f"  Conv1: 3→32 通道, 3x3 卷积")
    print(f"  Conv2: 32→64 通道, 3x3 卷积")
    print(f"  FC1: 4096→256")
    print(f"  FC2: 256→10")
    print(f"\n总参数量: {total_params:,}")
else:
    print("PyTorch 未安装，跳过模型创建")

In [ ]:
# ============================================================
# 导出为 ONNX 格式
# ============================================================

if PYTORCH_AVAILABLE:
    # 创建临时目录保存模型
    model_dir = tempfile.mkdtemp()
    onnx_path = os.path.join(model_dir, "model.onnx")
    
    # 创建示例输入
    dummy_input = torch.randn(1, 3, 32, 32)
    
    # 导出 ONNX 模型
    torch.onnx.export(
        model,                          # PyTorch 模型
        dummy_input,                    # 示例输入 (用于追踪)
        onnx_path,                      # 输出路径
        input_names=['input'],          # 输入节点名称
        output_names=['output'],        # 输出节点名称
        dynamic_axes={                  # 动态维度配置
            'input': {0: 'batch_size'},   # batch 维度可变
            'output': {0: 'batch_size'}
        },
        opset_version=14                # ONNX 算子集版本
    )
    
    print("=" * 60)
    print("ONNX 导出完成")
    print("=" * 60)
    print(f"\n保存路径: {onnx_path}")
    print(f"模型大小: {os.path.getsize(onnx_path) / 1024:.2f} KB")
else:
    print("PyTorch 未安装，跳过 ONNX 导出")

<cell_type>markdown</cell_type>## 3. 构建 TensorRT 引擎

**核心概念**: TensorRT 引擎构建需要以下组件

```
┌─────────────────────────────────────────────────────────────┐
│                    TensorRT 核心组件                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Logger (日志记录器)                                        │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  记录构建和推理过程中的信息                         │   │
│  │  级别: VERBOSE, INFO, WARNING, ERROR, INTERNAL_ERROR│   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  Builder (引擎构建器)                                       │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  负责构建优化后的引擎                               │   │
│  │  - 创建 Network                                    │   │
│  │  - 配置优化选项                                    │   │
│  │  - 执行优化和编译                                  │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  Network (网络定义)                                         │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  表示神经网络的计算图                               │   │
│  │  - 输入/输出张量                                   │   │
│  │  - 层定义                                          │   │
│  │  - 连接关系                                        │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  BuilderConfig (构建配置)                                   │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  配置优化选项                                       │   │
│  │  - 精度模式 (FP32/FP16/INT8)                       │   │
│  │  - 工作空间大小                                    │   │
│  │  - 动态形状配置                                    │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 步骤 1: 创建 Logger 和 Builder
# ============================================================
if TENSORRT_AVAILABLE and PYTORCH_AVAILABLE:
    # 创建 Logger (日志记录器)
    # 级别: VERBOSE, INFO, WARNING, ERROR, INTERNAL_ERROR
    logger = trt.Logger(trt.Logger.WARNING)
    
    # 创建 Builder (引擎构建器)
    builder = trt.Builder(logger)
    
    # 创建网络定义 (显式批次模式)
    # EXPLICIT_BATCH: 批次大小作为网络的一部分，支持动态批次
    network_flags = 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    network = builder.create_network(network_flags)
    
    # 创建 ONNX 解析器
    parser = trt.OnnxParser(network, logger)
    
    # 解析 ONNX 模型
    with open(onnx_path, "rb") as f:
        parse_success = parser.parse(f.read())
    
    print("=" * 60)
    print("ONNX 解析结果")
    print("=" * 60)
    
    if parse_success:
        print(f"\n✓ ONNX 解析成功!")
        print(f"  网络输入数: {network.num_inputs}")
        print(f"  网络输出数: {network.num_outputs}")
        print(f"  网络层数: {network.num_layers}")
        
        # 显示输入输出信息
        print(f"\n输入张量:")
        for i in range(network.num_inputs):
            inp = network.get_input(i)
            print(f"  {inp.name}: {inp.shape}")
        
        print(f"\n输出张量:")
        for i in range(network.num_outputs):
            out = network.get_output(i)
            print(f"  {out.name}: {out.shape}")
    else:
        print(f"\n✗ ONNX 解析失败!")
        for i in range(parser.num_errors):
            print(f"  错误 {i+1}: {parser.get_error(i)}")
else:
    print("跳过 TensorRT 示例 (TensorRT 或 PyTorch 未安装)")

In [ ]:
# ============================================================
# 步骤 2: 配置构建选项
# ============================================================
if TENSORRT_AVAILABLE and PYTORCH_AVAILABLE:
    # 创建构建配置
    config = builder.create_builder_config()
    
    # ============================================================
    # 设置工作空间大小 (用于优化算法的临时内存)
    # ============================================================
    # 1GB 工作空间，更大的空间允许更多优化选项
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
    
    # ============================================================
    # 启用 FP16 精度 (如果 GPU 支持)
    # ============================================================
    if builder.platform_has_fast_fp16:
        config.set_flag(trt.BuilderFlag.FP16)
        print("✓ 启用 FP16 精度 (GPU 支持快速 FP16)")
    else:
        print("! FP16 不可用，使用 FP32")
    
    # ============================================================
    # 配置动态形状 (Optimization Profile)
    # ============================================================
    # 动态形状需要指定: 最小、最优、最大形状
    profile = builder.create_optimization_profile()
    profile.set_shape(
        "input",                    # 输入名称
        min=(1, 3, 32, 32),        # 最小形状 (batch=1)
        opt=(8, 3, 32, 32),        # 最优形状 (batch=8，TensorRT 会针对此优化)
        max=(32, 3, 32, 32)        # 最大形状 (batch=32)
    )
    config.add_optimization_profile(profile)
    
    print("=" * 60)
    print("构建配置完成")
    print("=" * 60)
    print(f"\n工作空间大小: 1 GB")
    print(f"动态批次范围: 1 - 32")
    print(f"最优批次大小: 8")
else:
    print("跳过 TensorRT 示例")

In [ ]:
# ============================================================
# 步骤 3: 构建 TensorRT 引擎
# ============================================================
if TENSORRT_AVAILABLE and PYTORCH_AVAILABLE:
    print("=" * 60)
    print("构建 TensorRT 引擎")
    print("=" * 60)
    print("\n正在构建引擎 (可能需要几分钟)...")
    print("TensorRT 会尝试不同的优化策略并选择最优方案")
    
    # 构建序列化引擎
    serialized_engine = builder.build_serialized_network(network, config)
    
    if serialized_engine:
        print(f"\n✓ 引擎构建成功!")
        print(f"  引擎大小: {len(serialized_engine) / 1024:.2f} KB")
        
        # 保存引擎到文件
        engine_path = os.path.join(model_dir, "model.engine")
        with open(engine_path, "wb") as f:
            f.write(serialized_engine)
        print(f"  保存路径: {engine_path}")
        
        # 比较大小
        onnx_size = os.path.getsize(onnx_path) / 1024
        engine_size = len(serialized_engine) / 1024
        print(f"\n大小对比:")
        print(f"  ONNX 模型: {onnx_size:.2f} KB")
        print(f"  TensorRT 引擎: {engine_size:.2f} KB")
    else:
        print("\n✗ 引擎构建失败!")
        engine_path = None
else:
    print("跳过 TensorRT 示例")
    engine_path = None

<cell_type>markdown</cell_type>## 4. TensorRT 推理执行

**核心概念**: 使用构建好的引擎进行推理

```
┌─────────────────────────────────────────────────────────────┐
│                    TensorRT 推理流程                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 加载序列化引擎                                          │
│     ┌─────────────────────────────────────────────────┐    │
│     │  runtime = trt.Runtime(logger)                  │    │
│     │  engine = runtime.deserialize_cuda_engine(data) │    │
│     └─────────────────────────────────────────────────┘    │
│                                                             │
│  2. 创建执行上下文                                          │
│     ┌─────────────────────────────────────────────────┐    │
│     │  context = engine.create_execution_context()    │    │
│     │  # 上下文管理推理状态                           │    │
│     └─────────────────────────────────────────────────┘    │
│                                                             │
│  3. 分配 GPU 内存                                           │
│     ┌─────────────────────────────────────────────────┐    │
│     │  d_input = cuda.mem_alloc(input_size)           │    │
│     │  d_output = cuda.mem_alloc(output_size)         │    │
│     └─────────────────────────────────────────────────┘    │
│                                                             │
│  4. 执行推理                                                │
│     ┌─────────────────────────────────────────────────┐    │
│     │  cuda.memcpy_htod(d_input, h_input)  # CPU→GPU │    │
│     │  context.execute_v2(bindings)         # 推理    │    │
│     │  cuda.memcpy_dtoh(h_output, d_output) # GPU→CPU │    │
│     └─────────────────────────────────────────────────┘    │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# TensorRT 推理执行
# ============================================================
if TENSORRT_AVAILABLE and PYCUDA_AVAILABLE and engine_path and os.path.exists(engine_path):
    print("=" * 60)
    print("TensorRT 推理执行")
    print("=" * 60)
    
    # 加载序列化引擎
    runtime = trt.Runtime(logger)
    with open(engine_path, "rb") as f:
        engine = runtime.deserialize_cuda_engine(f.read())
    
    # 创建执行上下文
    context = engine.create_execution_context()
    
    # 设置输入形状 (动态形状需要)
    batch_size = 8
    context.set_input_shape("input", (batch_size, 3, 32, 32))
    
    # 准备输入数据
    h_input = np.random.randn(batch_size, 3, 32, 32).astype(np.float32)
    h_output = np.empty((batch_size, 10), dtype=np.float32)
    
    # 分配 GPU 内存
    d_input = cuda.mem_alloc(h_input.nbytes)
    d_output = cuda.mem_alloc(h_output.nbytes)
    
    # 创建 CUDA 流
    stream = cuda.Stream()
    
    # 执行推理
    # 1. 将输入数据从 CPU 复制到 GPU
    cuda.memcpy_htod_async(d_input, h_input, stream)
    
    # 2. 执行推理
    context.execute_async_v2(
        bindings=[int(d_input), int(d_output)],
        stream_handle=stream.handle
    )
    
    # 3. 将输出数据从 GPU 复制到 CPU
    cuda.memcpy_dtoh_async(h_output, d_output, stream)
    
    # 4. 同步等待完成
    stream.synchronize()
    
    print(f"\n✓ 推理执行成功!")
    print(f"  输入形状: {h_input.shape}")
    print(f"  输出形状: {h_output.shape}")
    print(f"  输出示例 (第一个样本): {h_output[0][:5].round(4)}")
else:
    print("跳过 TensorRT 推理示例 (依赖未安装或引擎未构建)")

In [ ]:
# ============================================================
# 验证 TensorRT 与 PyTorch 输出一致性
# ============================================================
if TENSORRT_AVAILABLE and PYCUDA_AVAILABLE and PYTORCH_AVAILABLE and engine_path:
    print("=" * 60)
    print("输出一致性验证")
    print("=" * 60)
    
    # 使用相同输入比较 PyTorch 和 TensorRT 的输出
    test_input = np.random.randn(batch_size, 3, 32, 32).astype(np.float32)
    
    # PyTorch 推理
    with torch.no_grad():
        pytorch_output = model(torch.from_numpy(test_input)).numpy()
    
    # TensorRT 推理
    h_input = test_input.copy()
    h_output = np.empty((batch_size, 10), dtype=np.float32)
    
    cuda.memcpy_htod(d_input, h_input)
    context.execute_v2(bindings=[int(d_input), int(d_output)])
    cuda.memcpy_dtoh(h_output, d_output)
    
    # 计算差异
    diff = np.abs(pytorch_output - h_output)
    
    print(f"\nPyTorch 输出 (第一个样本): {pytorch_output[0][:5].round(4)}")
    print(f"TensorRT 输出 (第一个样本): {h_output[0][:5].round(4)}")
    print(f"\n最大差异: {diff.max():.6f}")
    print(f"平均差异: {diff.mean():.6f}")
    print(f"\n输出一致: {'✓ 通过' if diff.max() < 0.01 else '! 差异较大 (FP16 精度)'}")
else:
    print("跳过输出一致性验证")

<cell_type>markdown</cell_type>## 5. 性能对比: PyTorch vs TensorRT

**核心概念**: 通过基准测试比较不同推理方式的性能

```
┌─────────────────────────────────────────────────────────────┐
│                    性能对比预期                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  推理方式              相对速度      适用场景               │
│  ─────────────────────────────────────────────────────────  │
│  PyTorch CPU           1x           开发调试               │
│  PyTorch GPU           5-10x        快速原型               │
│  TensorRT FP32         10-15x       精度敏感               │
│  TensorRT FP16         15-25x       生产推理 (推荐)        │
│  TensorRT INT8         20-40x       高吞吐场景             │
│                                                             │
│  注意: 实际加速比取决于模型结构和 GPU 型号                 │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# PyTorch 基准测试
# ============================================================

def benchmark_pytorch(model, input_data, num_runs=100, warmup=10):
    """
    PyTorch 推理基准测试
    
    参数:
        model: PyTorch 模型
        input_data: 输入张量
        num_runs: 测试运行次数
        warmup: 预热运行次数
        
    返回:
        mean_ms: 平均延迟 (毫秒)
        std_ms: 标准差 (毫秒)
    """
    model.eval()
    
    # 预热 (让系统缓存稳定)
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(input_data)
    
    # 计时
    latencies = []
    with torch.no_grad():
        for _ in range(num_runs):
            start = time.perf_counter()
            _ = model(input_data)
            latencies.append((time.perf_counter() - start) * 1000)
    
    return np.mean(latencies), np.std(latencies)


if PYTORCH_AVAILABLE:
    print("=" * 60)
    print("PyTorch 基准测试")
    print("=" * 60)
    
    # PyTorch CPU 基准
    input_tensor = torch.randn(8, 3, 32, 32)
    pytorch_cpu_mean, pytorch_cpu_std = benchmark_pytorch(model, input_tensor)
    print(f"\nPyTorch CPU: {pytorch_cpu_mean:.2f} ± {pytorch_cpu_std:.2f} ms")
    
    # PyTorch GPU 基准 (如果可用)
    if torch.cuda.is_available():
        model_gpu = model.cuda()
        input_gpu = input_tensor.cuda()
        
        # GPU 需要同步
        torch.cuda.synchronize()
        pytorch_gpu_mean, pytorch_gpu_std = benchmark_pytorch(model_gpu, input_gpu)
        torch.cuda.synchronize()
        
        print(f"PyTorch GPU: {pytorch_gpu_mean:.2f} ± {pytorch_gpu_std:.2f} ms")
        print(f"\nGPU vs CPU 加速比: {pytorch_cpu_mean / pytorch_gpu_mean:.2f}x")
    else:
        print("\nCUDA 不可用，跳过 GPU 测试")

# ============================================================
# TensorRT 基准测试
# ============================================================
if TENSORRT_AVAILABLE and PYCUDA_AVAILABLE and engine_path:
    def benchmark_tensorrt(context, d_input, d_output, h_input, h_output, 
                           num_runs=100, warmup=10):
        """
        TensorRT 推理基准测试
        
        参数:
            context: TensorRT 执行上下文
            d_input: GPU 输入缓冲区
            d_output: GPU 输出缓冲区
            h_input: CPU 输入数据
            h_output: CPU 输出缓冲区
            num_runs: 测试运行次数
            warmup: 预热运行次数
            
        返回:
            mean_ms: 平均延迟 (毫秒)
            std_ms: 标准差 (毫秒)
        """
        # 预热
        for _ in range(warmup):
            cuda.memcpy_htod(d_input, h_input)
            context.execute_v2(bindings=[int(d_input), int(d_output)])
            cuda.memcpy_dtoh(h_output, d_output)
        
        # 计时
        latencies = []
        for _ in range(num_runs):
            start = time.perf_counter()
            cuda.memcpy_htod(d_input, h_input)
            context.execute_v2(bindings=[int(d_input), int(d_output)])
            cuda.memcpy_dtoh(h_output, d_output)
            latencies.append((time.perf_counter() - start) * 1000)
        
        return np.mean(latencies), np.std(latencies)
    
    print("=" * 60)
    print("TensorRT 基准测试")
    print("=" * 60)
    
    # TensorRT 基准
    trt_mean, trt_std = benchmark_tensorrt(
        context, d_input, d_output, h_input, h_output
    )
    print(f"\nTensorRT FP16: {trt_mean:.2f} ± {trt_std:.2f} ms")
    
    # 计算加速比
    print(f"\n性能对比:")
    print(f"  TensorRT vs PyTorch CPU: {pytorch_cpu_mean / trt_mean:.2f}x")
    if torch.cuda.is_available():
        print(f"  TensorRT vs PyTorch GPU: {pytorch_gpu_mean / trt_mean:.2f}x")
else:
    print("跳过 TensorRT 基准测试 (依赖未安装)")

In [ ]:
# ============================================================
# 性能对比可视化
# ============================================================
if PYTORCH_AVAILABLE:
    try:
        import matplotlib.pyplot as plt
        
        # 准备数据
        methods = ['PyTorch CPU']
        times = [pytorch_cpu_mean]
        
        if torch.cuda.is_available():
            methods.append('PyTorch GPU')
            times.append(pytorch_gpu_mean)
        
        if TENSORRT_AVAILABLE and PYCUDA_AVAILABLE and engine_path:
            methods.append('TensorRT FP16')
            times.append(trt_mean)
        
        # 绘制柱状图
        fig, ax = plt.subplots(figsize=(10, 5))
        bars = ax.bar(methods, times, color=['steelblue', 'coral', 'green'][:len(methods)], 
                      alpha=0.7, edgecolor='black')
        
        ax.set_ylabel('Latency (ms)')
        ax.set_title('推理延迟对比 (batch_size=8)', fontsize=12)
        ax.grid(axis='y', alpha=0.3)
        
        # 添加数值标签
        for bar, time_val in zip(bars, times):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                   f'{time_val:.2f}ms', ha='center', va='bottom', fontsize=10)
        
        plt.tight_layout()
        plt.show()
        
        print("\n观察:")
        print("  - TensorRT 通过层融合和内核优化实现显著加速")
        print("  - FP16 精度在保持精度的同时提供额外加速")
    except ImportError:
        print("matplotlib 未安装，跳过可视化")

In [ ]:
<cell_type>markdown</cell_type>## 6. INT8 量化 (高级优化)

**核心概念**: INT8 量化可以获得更高的加速，但需要校准数据

```
┌─────────────────────────────────────────────────────────────┐
│                    INT8 量化流程                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 准备校准数据集 (代表性数据，通常 100-1000 样本)        │
│                        ↓                                    │
│  2. 运行校准，收集激活值分布                                │
│     ┌─────────────────────────────────────────────────┐    │
│     │  校准算法:                                      │    │
│     │  - ENTROPY: 最小化 KL 散度 (推荐)              │    │
│     │  - MINMAX: 使用最小/最大值                     │    │
│     │  - PERCENTILE: 使用百分位数                    │    │
│     └─────────────────────────────────────────────────┘    │
│                        ↓                                    │
│  3. 计算量化参数 (scale, zero_point)                        │
│                        ↓                                    │
│  4. 构建 INT8 引擎                                          │
│                                                             │
│  INT8 优势:                                                 │
│  - 比 FP16 快 2x                                           │
│  - 内存占用减少 50%                                        │
│                                                             │
│  INT8 注意事项:                                             │
│  - 需要校准数据                                            │
│  - 可能有精度损失 (通常 <1%)                               │
│  - 校准数据要有代表性                                      │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

# ============================================================
# INT8 量化配置示例
# ============================================================
print("=" * 60)
print("INT8 量化配置示例")
print("=" * 60)

print("""
INT8 量化需要实现 Calibrator 类:

class MyCalibrator(trt.IInt8EntropyCalibrator2):
    def __init__(self, calibration_data, cache_file):
        super().__init__()
        self.data = calibration_data
        self.cache_file = cache_file
        self.current_index = 0
        
        # 分配 GPU 缓冲区
        self.device_input = cuda.mem_alloc(data[0].nbytes)
    
    def get_batch_size(self):
        return 1
    
    def get_batch(self, names):
        if self.current_index < len(self.data):
            # 将校准数据复制到 GPU
            cuda.memcpy_htod(self.device_input, self.data[self.current_index])
            self.current_index += 1
            return [int(self.device_input)]
        return None
    
    def read_calibration_cache(self):
        # 读取缓存的校准数据
        if os.path.exists(self.cache_file):
            with open(self.cache_file, 'rb') as f:
                return f.read()
        return None
    
    def write_calibration_cache(self, cache):
        # 保存校准数据到缓存
        with open(self.cache_file, 'wb') as f:
            f.write(cache)

# 使用 INT8 配置:
# config.set_flag(trt.BuilderFlag.INT8)
# config.int8_calibrator = MyCalibrator(calibration_data, "calibration.cache")
""")

print("\n校准算法选择:")
print("  - IInt8EntropyCalibrator2: 最小化 KL 散度 (推荐)")
print("  - IInt8MinMaxCalibrator: 使用最小/最大值")
print("  - IInt8LegacyCalibrator: 旧版校准器")

In [ ]:
# ============================================================
# 动态形状测试
# ============================================================
if TENSORRT_AVAILABLE and PYCUDA_AVAILABLE and engine_path:
    print("=" * 60)
    print("动态形状测试")
    print("=" * 60)
    
    # 测试不同批次大小
    test_batch_sizes = [1, 4, 8, 16, 32]
    
    print(f"\n使用同一个引擎测试不同批次大小:")
    print(f"{'Batch Size':<15} {'输入形状':<25} {'输出形状':<20}")
    print("-" * 60)
    
    for batch_size in test_batch_sizes:
        # 设置输入形状
        context.set_input_shape("input", (batch_size, 3, 32, 32))
        
        # 准备数据
        h_input = np.random.randn(batch_size, 3, 32, 32).astype(np.float32)
        h_output = np.empty((batch_size, 10), dtype=np.float32)
        
        # 重新分配缓冲区
        d_input = cuda.mem_alloc(h_input.nbytes)
        d_output = cuda.mem_alloc(h_output.nbytes)
        
        # 执行推理
        cuda.memcpy_htod(d_input, h_input)
        context.execute_v2(bindings=[int(d_input), int(d_output)])
        cuda.memcpy_dtoh(h_output, d_output)
        
        print(f"{batch_size:<15} {str(h_input.shape):<25} {str(h_output.shape):<20}")
    
    print(f"\n✓ 动态形状工作正常!")
    print(f"  同一个 TensorRT 引擎可以处理 1-32 批次大小的输入")
else:
    print("跳过动态形状测试")

<cell_type>markdown</cell_type>## 总结

本教程介绍了 TensorRT 的核心功能和最佳实践：

### 核心知识点

| 主题 | 关键内容 |
|:-----|:---------|
| 优化技术 | 层融合、内核自动调优、内存优化 |
| 精度模式 | FP32 (基准)、FP16 (2x加速)、INT8 (4x加速) |
| 引擎构建 | Logger → Builder → Network → Config → Engine |
| 动态形状 | Optimization Profile 配置 min/opt/max |
| 推理执行 | Context、GPU 内存分配、异步执行 |

### TensorRT API 速查

```python
import tensorrt as trt

# 1. 创建 Logger 和 Builder
logger = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(logger)

# 2. 创建网络并解析 ONNX
network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
parser = trt.OnnxParser(network, logger)
parser.parse(onnx_data)

# 3. 配置构建选项
config = builder.create_builder_config()
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
config.set_flag(trt.BuilderFlag.FP16)  # 启用 FP16

# 4. 配置动态形状
profile = builder.create_optimization_profile()
profile.set_shape("input", min=(1,3,32,32), opt=(8,3,32,32), max=(32,3,32,32))
config.add_optimization_profile(profile)

# 5. 构建引擎
serialized_engine = builder.build_serialized_network(network, config)

# 6. 推理
runtime = trt.Runtime(logger)
engine = runtime.deserialize_cuda_engine(serialized_engine)
context = engine.create_execution_context()
context.execute_v2(bindings=[d_input, d_output])
```

### 最佳实践

```
TensorRT 优化检查清单:
✓ 使用 FP16 精度 (大多数场景推荐)
✓ 配置合适的工作空间大小
✓ 设置正确的动态形状范围
✓ 保存引擎文件避免重复构建
✓ 使用异步执行和 CUDA 流
✓ INT8 量化需要准备校准数据

常见问题:
✗ 引擎构建时间长 → 保存引擎文件复用
✗ 动态形状超出范围 → 检查 profile 配置
✗ 精度损失 → 检查 FP16/INT8 敏感层
✗ 内存不足 → 减小工作空间或批次大小
```

### 下一步学习

- **03_vLLM_tutorial.ipynb**: LLM 专用推理引擎
- **04_Advanced_Inference_tutorial.ipynb**: 高级推理技术

In [ ]:
# ============================================================
# 清理临时文件
# ============================================================
import shutil

# 清理临时目录
if PYTORCH_AVAILABLE and 'model_dir' in dir():
    shutil.rmtree(model_dir, ignore_errors=True)

print("=" * 60)
print("清理完成")
print("=" * 60)
print("\n✓ 临时文件已清理")
print("\n本教程演示了 TensorRT 的核心功能:")
print("  1. TensorRT 优化技术 (层融合、内核调优)")
print("  2. 从 ONNX 构建 TensorRT 引擎")
print("  3. 精度模式 (FP32/FP16/INT8)")
print("  4. 动态形状配置")
print("  5. 推理执行和性能对比")